# atomic-gdn2 — Kaggle/Colab TPU runner
Thin launcher: the notebook never imports JAX (so it never holds the TPU); training runs in a subprocess
of an isolated venv with the pinned TPU stack. All logic lives in the `atomic_gdn2` package.

1. Set `EXPERIMENT` below (`baseline_gdn2_owt` or `hybrid_hier_1024_28l`).
2. Kaggle: Accelerator = TPU VM v5e-8/v3-8, add dataset `windmaple/openwebtext-gpt2`, internet on.
3. Add/enable Kaggle Secret `GITHUB_TOKEN` for private repository access.

3. Add/enable Kaggle Secret `GITHUB_TOKEN` for private repository access.


In [ ]:
EXPERIMENT = "hybrid_hier_1024_28l"      # or "baseline_gdn2_owt"
RESUME_FROM = None                        # e.g. "/kaggle/input/<notebook>/<run>/checkpoints"
OVERRIDES = []                            # e.g. ["train.epochs=1"]
REPO_URL = "https://github.com/Akseleu-J/atomic-gdn2.git"
REPO_REF = "main"


In [ ]:
import os, subprocess, sys, base64, time
from pathlib import Path
JOB_DEADLINE_UNIX = time.time() + 32040
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content") if Path("/content").exists() else Path.cwd()
REPO = WORK / "atomic-gdn2"
github_token = os.environ.get("GITHUB_TOKEN")
if not github_token:
    from kaggle_secrets import UserSecretsClient
    github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("GITHUB_TOKEN is required for the private repository")
git_env = os.environ.copy()
git_env.update(GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="2",
    GIT_CONFIG_KEY_0="credential.helper", GIT_CONFIG_VALUE_0="",
    GIT_CONFIG_KEY_1="http.https://github.com/.extraheader",
    GIT_CONFIG_VALUE_1="AUTHORIZATION: basic " + base64.b64encode(("x-access-token:" + github_token).encode()).decode())
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)], env=git_env, check=True)
del github_token, git_env


In [ ]:
# Creates <WORK>/venv_train with jax[tpu]==0.11.2, flax, optax, orbax, atomic-ops and installs this repo (~5-10 min first time).
out = subprocess.run([sys.executable, str(REPO / "scripts" / "bootstrap_env.py")], capture_output=True, text=True)
print(out.stdout[-3000:], out.stderr[-3000:])
out.check_returncode()
VENV_PY = out.stdout.strip().splitlines()[-1]


In [ ]:
# Training in a subprocess; JAX_PLATFORMS / XLA_FLAGS must not force CPU.
env = {k: v for k, v in os.environ.items() if k not in ("JAX_PLATFORMS", "XLA_FLAGS")}
env["JAX_PLATFORMS"] = "tpu"
env["JAX_PLATFORMS"] = "tpu"
env["ATOMIC_LM_WORK_DIR"] = str(WORK / "runs")
cmd = [VENV_PY, "-u", "-m", "atomic_gdn2_model.cli", "--config", str(REPO / "configs" / f"{EXPERIMENT}.yaml")]
cmd += ["--set", f"runtime.wall_deadline_unix={JOB_DEADLINE_UNIX}", "--set", "runtime.checkpoint_margin_s=600"]
for o in OVERRIDES: cmd += ["--set", o]
if RESUME_FROM: cmd += ["--resume-from", RESUME_FROM]
print(" ".join(cmd))
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO)
for line in proc.stdout: print(line, end="")
assert proc.wait() == 0, f"training exited with {proc.returncode}"


Outputs: `<WORK>/runs/<name>/{config.yaml, metrics.jsonl, checkpoints/}`. Keep `checkpoints/` as notebook output to resume later via `RESUME_FROM`.